# Maximum optimization — full candidate

Private experiment based on verified version363823158. New internal metric-lifetime and frozen-NF4 MLP cast-movement paths preserve public outputs, training configuration, precision rounding, data, and every harness/test/task file. No measured benefit is claimed until this saved version passes tests and the full unchanged harness. Organizer SwiGLU kernels remain attributed/excluded from own share. Optional frozen-head recomputation is diagnostic only; stride-aware accuracy is True.


In [ ]:
DEVELOPMENT_SNAPSHOT_SHA256="ddc7740aa180e81cbdafa886d4fa44c1698cd2f79ae1cc45e63dbd45bedb033b"
print({"mode":"maxopt_full","snapshot_sha256":DEVELOPMENT_SNAPSHOT_SHA256})


In [ ]:
import os, subprocess
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
# Remove an incompatible optional package before all tests and measurements.
!python -m pip uninstall -y torchao
!python -m pip install -q "soup-cli[train]==0.77.0" "transformers==5.16.1" "peft==0.20.0" "trl==0.29.1" "bitsandbytes==0.50.2" "accelerate==1.14.0" pytest

In [ ]:
import base64, hashlib, io, json, shutil, zipfile
from pathlib import Path

# Development source snapshot, not a GitHub clone or a final submission commit.
SNAPSHOT_SHA256 = "ddc7740aa180e81cbdafa886d4fa44c1698cd2f79ae1cc45e63dbd45bedb033b"
STARTER_COMMIT = "c7c2fcd15972ce8b898d8d3ff732ee192c817ceb"
SOURCE_SNAPSHOT_B64 = (
    "..."
    "AAA="
)
payload = base64.b64decode(SOURCE_SNAPSHOT_B64, validate=True)
assert hashlib.sha256(payload).hexdigest() == SNAPSHOT_SHA256
repo = Path("/kaggle/working/repo")
assert repo.resolve() == Path("/kaggle/working/repo")
if repo.exists():
    shutil.rmtree(repo)
repo.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(payload)) as source:
    for entry in source.infolist():
        target = (repo / entry.filename).resolve()
        assert repo.resolve() in target.parents, "Unsafe snapshot entry"
        assert not entry.is_dir()
    source.extractall(repo)
    snapshot_files = len(source.infolist())
(Path("/kaggle/working") / "development_snapshot.json").write_text(
    json.dumps({"source": "embedded development snapshot", "sha256": SNAPSHOT_SHA256,
                "starter_commit": STARTER_COMMIT, "files": snapshot_files}, indent=2))
del SOURCE_SNAPSHOT_B64, payload
print({"source": "development snapshot", "sha256": SNAPSHOT_SHA256,
       "starter_commit": STARTER_COMMIT, "files": snapshot_files})


In [ ]:
import os, subprocess, sys
sys.path.insert(0, "/kaggle/working/repo/harness")
from common import ENV

def run_repo(arguments, **overrides):
    environment = dict(ENV, **overrides)
    subprocess.run([sys.executable, *arguments], cwd="/kaggle/working/repo",
                   env=environment, check=True)

# Metadata inspection does not create a CUDA context in the notebook process.
import importlib.metadata
print({name: importlib.metadata.version(name) for name in
       ("soup-cli", "torch", "triton", "transformers", "peft", "trl", "bitsandbytes", "accelerate")})


## Separate correctness and synthetic checks

These are independent processes, using the harness's original GPU visibility and allocator environment. They terminate before the official run, leaving no notebook GPU allocations. Synthetic latency and allocation figures are not training score metrics.


In [ ]:
run_repo(["-m", "pytest", "solution", "-q"])
run_repo(["-m", "pytest", "tests", "-q"], SOUP_APPLY_SOLUTION="1")


In [ ]:
run_repo(["-m", "solution.benchmark", "--out", "/kaggle/working/development_microbench.json"])
run_repo(["-m","solution.benchmark_half","--out","/kaggle/working/half_ce_microbenchmark.json"])


In [ ]:
run_repo(["task/prepare_data.py"])


## Original official measurement commands

The unchanged harness downloads the model in its warmup process, then starts each measured run in a separate process with empty compiler caches. `--repeats 2` selects the documented jury order. The rule checker and score use the resulting `runs/result.json`.


In [ ]:
run_repo(["harness/run_task.py", "--repeats", "2"])


In [ ]:
run_repo(["harness/check_rules.py"])
run_repo(["harness/score.py", "runs/result.json"])


In [ ]:
import json
from pathlib import Path
result = json.loads(Path("/kaggle/working/repo/runs/result.json").read_text())
checks = json.loads(Path("/kaggle/working/repo/runs/checks.json").read_text())
print(json.dumps({"snapshot_sha256": DEVELOPMENT_SNAPSHOT_SHA256,
                  "summary": result.get("summary"), "checks": checks}, indent=2))


Outputs: `repo/runs/result.json`, `repo/runs/checks.json`, per-run logs and adapters, `development_microbench.json`, and `development_snapshot.json`. Preserve them with the saved Kaggle version. Failed checks must be investigated; neither the snapshot nor a synthetic benchmark establishes admission.
